# 06 - Evaluation: all three tiers on all 15 questions

This is where the project's result comes from. Every question goes through
Tier 1, Tier 2 and Tier 3, scored the same way.

The two gaps are what we are measuring:

- **Tier 1 to Tier 2** - what using real data is worth
- **Tier 2 to Tier 3** - what checking is worth

This takes a while on a laptop: 15 questions x 3 tiers, plus Tier 3 retries.


## Setup

All the code from notebooks 04 and 05, so this notebook runs on its own.


In [ ]:
import json
import re
import time

import duckdb
import sqlglot
from sqlglot import expressions as exp

DB = "../data/processed/evidenceiq.duckdb"
MODEL = "gemma3:4b"          # Abdelateef used gemma3:12b - use what you have
TOLERANCE = 0.01


# We never let the agent write to the database. Two safety nets: this check,
# and opening DuckDB with read_only=True.
#
# We parse the SQL into a syntax tree instead of searching for banned words.
# Word matching is easy to fool - a column called "updated_at" contains
# "update" - and it misses things a parser catches for free.

FORBIDDEN = {"insert", "update", "delete", "create", "drop", "alter", "copy",
             "command", "merge", "truncate", "attach", "detach", "install",
             "load", "pragma"}


def check_sql(sql):
    """Return None if the query is safe, otherwise a reason why it is not."""
    if not sql or not sql.strip():
        return "the query is empty"

    try:
        statements = sqlglot.parse(sql, read="duckdb")
    except Exception as e:
        return "could not parse the SQL: %s" % e

    if len(statements) != 1:
        return "only one statement is allowed"

    tree = statements[0]

    for node in tree.walk():
        key = getattr(node, "key", "").lower()
        if key in FORBIDDEN:
            return "forbidden operation: %s" % key

    if tree.find(exp.Select) is None:
        return "only SELECT queries are allowed"

    return None


def add_to_log(log, tool, code, ok, rows=None, error=None):
    """Every tool call goes in the log. The verifier only trusts what's here."""
    call = {"n": len(log), "tool": tool, "code": code,
            "ok": ok, "rows": rows or [], "error": error}
    log.append(call)
    return call


def numbers_in_log(log):
    """Every number the tools actually returned, with which call it came from."""
    found = []
    for call in log:
        if not call["ok"]:
            continue                      # a failed query proves nothing
        for row in call["rows"]:
            for value in row.values():
                if isinstance(value, (int, float)) and not isinstance(value, bool):
                    found.append((call["n"], float(value)))
    return found


def show_results(log, start=0):
    """Turn the log into text we can paste into the next prompt."""
    if len(log) <= start:
        return "(no queries were run)"
    text = ""
    for call in log[start:]:
        text += "[call %d] %s\n%s\n" % (call["n"], call["tool"], str(call["code"]).strip())
        text += "-> %s\n\n" % (json.dumps(call["rows"][:20]) if call["ok"]
                                else "FAILED: " + str(call["error"]))
    return text


def sql_metadata(sql):
    """Which columns and filters did this query actually use?

    Two of the seven things we have to show the user are the fields and the
    filters. We read them off the query itself rather than asking the model,
    because the model will happily make them up.

    Aliases are skipped: in "SUM(revenue) AS total_revenue" the real field is
    revenue, not total_revenue.
    """
    try:
        tree = sqlglot.parse_one(sql, read="duckdb")
    except Exception:
        return [], []

    aliases = {a.alias for a in tree.find_all(exp.Alias) if a.alias}

    fields = []
    for column in tree.find_all(exp.Column):
        name = column.name
        if name and name not in aliases and name not in fields:
            fields.append(name)

    filters = []
    for where in tree.find_all(exp.Where):
        text = where.this.sql(dialect="duckdb")
        if text not in filters:
            filters.append(text)

    return fields, filters


def json_safe(value):
    """Database values that json.dumps cannot handle."""
    if hasattr(value, "isoformat"):          # dates and timestamps
        return value.isoformat()
    if hasattr(value, "item"):               # numpy numbers
        return value.item()
    return value


def run_sql(sql, log):
    """Run one SELECT and add the result to the log."""
    problem = check_sql(sql)
    if problem:
        return add_to_log(log, "run_sql", sql, False, error=problem)

    try:
        con = duckdb.connect(DB, read_only=True)
        cursor = con.execute(sql)
        columns = [d[0] for d in cursor.description]
        rows = [{c: json_safe(v) for c, v in zip(columns, row)}
                for row in cursor.fetchmany(50)]
        con.close()
        return add_to_log(log, "run_sql", sql, True, rows)
    except Exception as e:
        return add_to_log(log, "run_sql", sql, False, error=str(e))


# The model does NOT get to write Python. It picks an operation from this
# list and gives us the numbers; we do the arithmetic ourselves.
#
# This is stricter than letting it write code, and it is also easier to check:
# the operation and its inputs are recorded in the log, so Tier 3 can redo the
# sum without trusting anything the model said.

OPERATIONS = ["pct_change", "difference", "ratio", "share", "sum", "mean"]


def calculate(operation, values):
    """Do the arithmetic. Returns (result, how_we_worked_it_out)."""
    if operation == "pct_change":
        if len(values) != 2:
            raise ValueError("pct_change needs [new, old]")
        new, old = values
        if old == 0:
            raise ValueError("cannot work out a percentage change from zero")
        return (new - old) / old * 100, "(%s - %s) / %s * 100" % (new, old, old)

    if operation == "difference":
        if len(values) != 2:
            raise ValueError("difference needs [a, b]")
        return values[0] - values[1], "%s - %s" % (values[0], values[1])

    if operation == "ratio":
        if len(values) != 2 or values[1] == 0:
            raise ValueError("ratio needs [top, bottom] and bottom cannot be zero")
        return values[0] / values[1], "%s / %s" % (values[0], values[1])

    if operation == "share":
        if len(values) != 2 or values[1] == 0:
            raise ValueError("share needs [part, total] and total cannot be zero")
        return values[0] / values[1] * 100, "%s / %s * 100" % (values[0], values[1])

    if operation == "sum":
        if not values:
            raise ValueError("sum needs at least one number")
        return sum(values), " + ".join(str(v) for v in values)

    if operation == "mean":
        if not values:
            raise ValueError("mean needs at least one number")
        return sum(values) / len(values), "mean of %d numbers" % len(values)

    raise ValueError("unknown operation: %s" % operation)


def run_python(request, log):
    """request = {"operation": ..., "values": [...], "result_name": ..., "unit": ...}"""
    operation = request.get("operation")
    values = [float(v) for v in (request.get("values") or [])]
    unit = request.get("unit") or ("%" if operation in ("pct_change", "share") else None)

    try:
        result, code = calculate(operation, values)
    except Exception as e:
        return add_to_log(log, "run_python", "%s(%s)" % (operation, values),
                          False, error=str(e))

    return add_to_log(log, "run_python", code, True,
                      [{"result_name": request.get("result_name") or operation,
                        "value": round(result, 4),
                        "unit": unit,
                        "operation": operation,
                        "inputs": values}])


CHART_TYPES = ["bar", "line", "scatter", "table", "none"]


def make_chart(spec, log):
    """We don't draw the chart, we just check the spec makes sense."""
    if spec.get("type") not in CHART_TYPES:
        return add_to_log(log, "make_chart", json.dumps(spec), False,
                          error="chart type must be one of %s" % CHART_TYPES)

    if spec.get("type") != "none":
        for needed in ("x", "y"):
            if not spec.get(needed):
                return add_to_log(log, "make_chart", json.dumps(spec), False,
                                  error="a chart needs both x and y")

    return add_to_log(log, "make_chart", json.dumps(spec), True, [spec])


def ask_gemma(system, question, shape):
    """Ask the model for JSON of a particular shape.

    `shape` is a JSON schema. Ollama forces the reply to match it, which is a
    lot more reliable than asking nicely and hoping. If the reply still will
    not parse we try once more with a blunter instruction.
    """
    import ollama

    for attempt in range(2):
        prompt = question if attempt == 0 else question + """

YOUR LAST REPLY WAS NOT VALID JSON.
Reply with ONE complete JSON object matching the schema. Keep the text short.
Do not write anything outside the JSON object."""

        reply = ollama.chat(model=MODEL, format=shape,
                            options={"temperature": 0, "num_predict": 2048},
                            messages=[{"role": "system", "content": system},
                                      {"role": "user", "content": prompt}])
        try:
            return json.loads(reply["message"]["content"])
        except Exception:
            continue

    return {"broken_json": True}


# The shapes we ask the model to fill in. Keeping them small is deliberate -
# a 4B model fills in three fields reliably and fifteen fields badly.

PLAN_SHAPE = {
    "type": "object",
    "properties": {
        "sufficient_data": {"type": "boolean"},
        "reason": {"type": "string"},
        "steps": {"type": "array", "items": {
            "type": "object",
            "properties": {
                "step": {"type": "integer"},
                "tool": {"type": "string", "enum": ["run_sql", "run_python", "make_chart"]},
                "objective": {"type": "string"},
            },
            "required": ["step", "tool", "objective"]}},
    },
    "required": ["sufficient_data", "steps"],
}

SQL_SHAPE = {"type": "object",
             "properties": {"sql": {"type": "string"}},
             "required": ["sql"]}

PYTHON_SHAPE = {
    "type": "object",
    "properties": {
        "operation": {"type": "string", "enum": OPERATIONS},
        "values": {"type": "array", "items": {"type": "number"}},
        "result_name": {"type": "string"},
        "unit": {"type": "string"},
    },
    "required": ["operation", "values", "result_name"],
}

CHART_SHAPE = {
    "type": "object",
    "properties": {
        "type": {"type": "string", "enum": CHART_TYPES},
        "source_tool_call": {"type": "integer"},
        "x": {"type": "string"},
        "y": {"type": "string"},
        "title": {"type": "string"},
    },
    "required": ["type"],
}

ANSWER_SHAPE = {
    "type": "object",
    "properties": {
        "findings": {"type": "string"},
        "claims": {"type": "array", "items": {
            "type": "object",
            "properties": {
                "text": {"type": "string"},
                "value": {"type": "number"},
                "unit": {"type": "string"},
                "from_call": {"type": "integer"},
                "calc": {"type": "string",
                         "enum": ["none", "pct_change", "share", "sum", "diff",
                                  "difference", "ratio", "mean"]},
                "inputs": {"type": "array", "items": {"type": "number"}},
            },
            "required": ["text"]}},
        "kpis": {"type": "object"},
        "limitations": {"type": "string"},
        "insufficient_data": {"type": "boolean"},
    },
    "required": ["findings", "claims"],
}


SCHEMA = """
Database: a UK online gift wholesaler, Dec 2009 to 9 Dec 2011, 1,033,030 rows.

TABLE sales (one row per invoice line)
  invoice_no, stock_code, description, quantity, unit_price, customer_id,
  country, revenue (= quantity * unit_price), invoice_date, invoice_month,
  is_cancellation, is_product, is_outlier

TABLE dim_month   invoice_month, trading_days, net_revenue, gross_revenue,
                  is_complete_month
  net_revenue EXCLUDES cancellations (use this). gross_revenue includes them.
TABLE dim_product stock_code, description, units_sold, gross_revenue
TABLE dim_customer customer_id, country, first_order, last_order, orders, net_revenue

There is NO cost, profit, margin, discount, competitor or customer age data.
"""

RULES = """
RULES (the answer is wrong without these):
1. Revenue always excludes cancellations:  WHERE NOT is_cancellation
2. Product questions also need:            AND is_product AND NOT is_outlier
3. Group products by stock_code, and use mode(description) for the name.
4. For anything about months use dim_month (it has trading_days already).
5. December 2011 only has 8 trading days, so never compare it as a full month.
6. Units sold also needs quantity > 0.
7. Never say what CAUSED something, only what contributed to it.
8. If the data can't answer the question, say so instead of guessing.
"""

EXAMPLES = """
EXAMPLES

Total revenue in 2011:
  SELECT ROUND(SUM(revenue),2) AS revenue FROM sales
  WHERE NOT is_cancellation AND year(invoice_date) = 2011

Top 5 products in 2011:
  SELECT stock_code, mode(description) AS description, ROUND(SUM(revenue),2) AS revenue
  FROM sales
  WHERE NOT is_cancellation AND is_product AND NOT is_outlier
    AND year(invoice_date) = 2011
  GROUP BY stock_code ORDER BY revenue DESC LIMIT 5

Two months compared:
  SELECT invoice_month, ROUND(net_revenue,2) AS revenue, trading_days,
         is_complete_month
  FROM dim_month WHERE invoice_month IN ('2011-10','2011-11')
"""

SYSTEM = "You are a careful business data analyst.\n" + SCHEMA + RULES

PLAN_PROMPT = SYSTEM + """
Plan how to answer the question. Do NOT answer it - you have not seen any data.

List the steps in order. Each step uses one tool:
  run_sql      fetch numbers from the database
  run_python   work something out from numbers a query already returned
  make_chart   show the result

Set sufficient_data to false if the data cannot answer the question at all.
"""

ANSWER_PROMPT = SYSTEM + """
You asked for some queries and here are the results. Write the answer using
ONLY these numbers.

Every number in findings must also appear as a claim, with:
  from_call  which call it came from
  calc       none, unless you worked it out - then pct_change/share/sum/diff/ratio
  inputs     the numbers you worked it out from

Do not say what caused anything. Say what contributed.
"""


def sql_prompt(question, objective):
    return (SYSTEM + EXAMPLES
            + "\nQUESTION\n" + question
            + "\n\nWRITE ONE SELECT QUERY FOR THIS STEP ONLY\n" + objective)


def repair_prompt(question, objective, sql, error):
    return (SYSTEM + EXAMPLES
            + "\nQUESTION\n" + question
            + "\n\nSTEP\n" + objective
            + "\n\nTHIS QUERY FAILED\n" + sql
            + "\n\nERROR\n" + str(error)
            + "\n\nWrite one corrected SELECT query.")


def python_prompt(question, objective, log):
    return (SYSTEM
            + "\nQUESTION\n" + question
            + "\n\nSTEP\n" + objective
            + "\n\nNUMBERS THE QUERIES RETURNED\n" + show_results(log)
            + "\nPick the operation and give us the numbers. We do the arithmetic.")


def chart_prompt(question, objective, log):
    return (SYSTEM
            + "\nQUESTION\n" + question
            + "\n\nSTEP\n" + objective
            + "\n\nRESULTS SO FAR\n" + show_results(log)
            + "\nChoose a chart. source_tool_call is the call number to plot.")


def answer_prompt(question, log, feedback=""):
    text = ("QUESTION\n" + question
            + "\n\nTOOL RESULTS\n" + show_results(log))
    if feedback:
        text += "\n\nYOUR LAST ANSWER FAILED CHECKING\n" + feedback
    return text


def touches_incomplete_month(log):
    """Did any query return a month flagged as incomplete?"""
    for call in log:
        if not call["ok"]:
            continue
        for row in call["rows"]:
            if row.get("is_complete_month") is False:
                return True
    return False


def mentions_december_2011(question):
    text = question.lower()
    return ("december 2011" in text or "dec 2011" in text
            or "2011-12" in text)


def claims_from_log(log):
    """Build claims ourselves when the model forgets to.

    Tier 2 must never hand Tier 3 an answer with no claims - every number would
    then slip through unchecked. So if the model returns none, we make them from
    what the tools actually returned.
    """
    claims = []
    for call in log:
        if not call["ok"] or call["tool"] == "make_chart":
            continue
        for row in call["rows"]:
            for key, value in row.items():
                if isinstance(value, bool) or not isinstance(value, (int, float)):
                    continue
                claims.append({"text": "%s = %s" % (key, value),
                               "value": float(value),
                               "unit": row.get("unit"),
                               "from_call": call["n"],
                               "calc": "none",
                               "inputs": []})
    return claims


def tier2(question, log=None, feedback=""):
    """Plan, run the steps, then write the answer from the real rows.

    The model never sees the database and never writes Python. It says what it
    wants; we run it and record what happened.
    """
    log = log if log is not None else []
    start = time.time()
    first_call = len(log)

    def stop(findings, limitations=None, insufficient=False, plan_steps=None):
        return {"question": question, "tier": 2, "findings": findings,
                "claims": [], "kpis": {}, "fields_used": [], "filters_used": [],
                "chart": None, "insufficient_data": insufficient,
                "limitations": limitations, "log": log, "retries": 0,
                "plan": plan_steps or [], "seconds": round(time.time() - start, 1)}

    # ---- 1. plan
    ask = question if not feedback else question + "\n\nLast attempt failed:\n" + feedback
    plan = ask_gemma(PLAN_PROMPT, ask, PLAN_SHAPE)

    if plan.get("broken_json"):
        return stop("The planner did not return valid JSON.",
                    "the model did not return valid JSON")

    steps = sorted(plan.get("steps") or [], key=lambda s: s.get("step", 0))
    plan_text = ["%s: %s" % (s.get("tool"), s.get("objective")) for s in steps]

    if not plan.get("sufficient_data", True):
        reason = plan.get("reason") or "The data cannot answer this question."
        return stop(reason, reason, True, plan_text)

    if not steps:
        return stop("The planner produced no steps to run.",
                    "the planner marked the question answerable but gave no steps",
                    False, plan_text)

    # ---- 2. run the steps
    chart = None

    for step in steps[:6]:
        tool = step.get("tool")
        objective = step.get("objective", "")

        if tool == "run_sql":
            request = ask_gemma(SYSTEM, sql_prompt(question, objective), SQL_SHAPE)
            call = run_sql(request.get("sql", ""), log)

            # one repair attempt, with the error message
            if not call["ok"]:
                fix = ask_gemma(SYSTEM,
                                repair_prompt(question, objective,
                                              call["code"], call["error"]),
                                SQL_SHAPE)
                call = run_sql(fix.get("sql", ""), log)

            if not call["ok"]:
                return stop("The query could not be made to run.",
                            call["error"], False, plan_text)

        elif tool == "run_python":
            # Do not work out a month-over-month change when one of the months
            # is incomplete. December 2011 has 8 trading days.
            if touches_incomplete_month(log) or mentions_december_2011(question):
                continue
            request = ask_gemma(SYSTEM, python_prompt(question, objective, log),
                                PYTHON_SHAPE)
            run_python(request, log)

        elif tool == "make_chart":
            spec = ask_gemma(SYSTEM, chart_prompt(question, objective, log),
                             CHART_SHAPE)
            call = make_chart(spec, log)
            if call["ok"] and spec.get("type") != "none":
                chart = spec

    # ---- 3. refuse if nothing worked
    useful = [c for c in log[first_call:]
              if c["ok"] and c["tool"] in ("run_sql", "run_python")]
    if not useful:
        return stop("No query produced any evidence, so there is no answer to give.",
                    "no successful query or calculation", False, plan_text)

    # ---- 4. write the answer from the real rows
    reply = ask_gemma(ANSWER_PROMPT, answer_prompt(question, log, feedback),
                      ANSWER_SHAPE)

    if reply.get("broken_json"):
        return stop("The model did not return a usable answer.",
                    "the model did not return valid JSON", False, plan_text)

    claims = reply.get("claims") or []
    if not claims:
        claims = claims_from_log(log[first_call:])

    # ---- 5. fields and filters come from the SQL, not from the model
    fields, filters = [], []
    for call in log[first_call:]:
        if call["tool"] != "run_sql" or not call["ok"]:
            continue
        f, w = sql_metadata(call["code"])
        fields += [x for x in f if x not in fields]
        filters += [x for x in w if x not in filters]

    limitations = reply.get("limitations")
    if (touches_incomplete_month(log) or mentions_december_2011(question)) \
            and not limitations:
        limitations = ("December 2011 has only 8 trading days, so it is not "
                       "comparable with a complete month.")

    return {"question": question, "tier": 2,
            "findings": reply.get("findings", ""),
            "claims": claims,
            "kpis": reply.get("kpis") or {},
            "fields_used": fields or [],
            "filters_used": filters or [],
            "chart": chart,
            "insufficient_data": bool(reply.get("insufficient_data")),
            "limitations": limitations,
            "log": log, "retries": 0, "plan": plan_text,
            "seconds": round(time.time() - start, 1)}


def tier1(question):
    """No data access at all. This is our baseline - it makes numbers up."""
    start = time.time()
    reply = get_json(ask_gemma(TIER1_PROMPT, question))
    claims = []
    if reply.get("value") is not None:
        claims = [{"text": reply.get("answer", ""), "value": reply.get("value"),
                   "unit": reply.get("unit"), "from_call": None,
                   "calc": "none", "inputs": []}]
    return {"question": question, "tier": 1,
            "findings": reply.get("answer", ""),
            "claims": claims,
            "kpis": {}, "fields_used": [], "filters_used": [], "chart": None,
            "insufficient_data": bool(reply.get("insufficient_data")),
            "limitations": reply.get("limitations"),
            "log": [], "retries": 0,
            "seconds": round(time.time() - start, 1)}


def about_equal(a, b, tol=TOLERANCE):
    """Compare two numbers allowing a small relative difference."""
    if a == b:
        return True
    biggest = max(abs(a), abs(b))
    if biggest == 0:
        return True
    return abs(a - b) / biggest <= tol


def which_call(number, log_numbers):
    """Which tool call returned this number? None if no call did."""
    for call_number, value in log_numbers:
        if about_equal(float(number), value):
            return call_number
    return None


def as_numbers(values):
    """Keep only the numbers.

    The model is supposed to put plain numbers in `inputs`, but it does not
    always. We have seen lists of dicts and numbers written as strings. The
    checker must never crash on bad model output - bad output is the whole
    reason it exists.
    """
    out = []
    for v in values or []:
        if isinstance(v, bool):
            continue
        if isinstance(v, (int, float)):
            out.append(float(v))
        elif isinstance(v, str):
            try:
                out.append(float(v.replace(",", "").replace("GBP", "").strip()))
            except ValueError:
                pass
        elif isinstance(v, dict):                 # {"PARTY BUNTING": 98237.49}
            for x in v.values():
                if isinstance(x, (int, float)) and not isinstance(x, bool):
                    out.append(float(x))
    return out


def recalculate(calc, inputs):
    """Work the number out ourselves, so we can compare."""
    inputs = as_numbers(inputs)
    try:
        if calc == "pct_change" and len(inputs) == 2:
            new, old = inputs
            return (new - old) / old * 100
        if calc == "share" and len(inputs) == 2:
            part, total = inputs
            return part / total * 100
        if calc == "diff" and len(inputs) == 2:
            return inputs[0] - inputs[1]
        if calc == "ratio" and len(inputs) == 2:
            return inputs[0] / inputs[1]
        if calc == "sum" and inputs:
            return sum(inputs)
    except (ZeroDivisionError, TypeError, ValueError):
        return None
    return None


def verify(answer):
    """Check every claim. Adds status / evidence / problem to each one.

    CHECK 1  is the number actually in the log?
    CHECK 2  if it was calculated, does it recalculate the same?
    CHECK 2b the model did the maths in its head, but got it right
    CHECK 3  do the parts add up to the total?
    CHECK 4  are there numbers in the paragraph that were never declared?
    """
    log = answer.get("log", [])
    log_numbers = numbers_in_log(log)

    for claim in answer.get("claims", []):
        # the model doesn't get to mark its own homework
        claim["status"] = None
        claim["evidence"] = None
        claim["problem"] = None

        value = claim.get("value")
        calc = claim.get("calc", "none")
        inputs = as_numbers(claim.get("inputs"))
        calculated = calc not in (None, "none") and inputs and value is not None

        # ---- CHECK 1: is this number in the log?
        if value is None:
            claim["status"] = "supported"          # a sentence with no number
        else:
            found = which_call(value, log_numbers)
            if found is not None:
                claim["status"] = "supported"
                claim["from_call"] = found
                claim["evidence"] = log[found]["code"]
            else:
                claim["status"] = "unsupported"
                claim["problem"] = "this number is not in any query result"

        # ---- CHECK 2: if it was calculated, redo the maths
        if calculated:
            expected = recalculate(calc, inputs)
            if expected is not None and not about_equal(float(value), expected):
                note = "we get %.2f, not %.2f" % (expected, float(value))
                if claim["status"] == "unsupported":
                    claim["problem"] = claim["problem"] + "; " + note
                else:
                    claim["status"] = "flagged"
                    claim["problem"] = note

        # ---- CHECK 2b: the model worked it out in its head and got it right.
        # If every input came from a query and our own maths agrees, the claim
        # is still traceable, so we accept it. Without this rule a CORRECT
        # answer gets hidden just because the model skipped run_python.
        if claim["status"] == "unsupported" and calculated:
            expected = recalculate(calc, inputs)
            from_calls = [which_call(x, log_numbers) for x in inputs]
            if (expected is not None
                    and about_equal(float(value), expected)
                    and all(c is not None for c in from_calls)):
                claim["status"] = "supported"
                claim["problem"] = None
                claim["from_call"] = from_calls[0]
                claim["evidence"] = ("we recalculated this ourselves from call %d:\n%s"
                                     % (from_calls[0], log[from_calls[0]]["code"]))

    # ---- CHECK 3: do the pieces add up?
    for problem in check_totals(answer):
        answer["limitations"] = ((answer.get("limitations") or "")
                                 + " " + problem).strip()

    # ---- CHECK 4: numbers in the paragraph that were never declared
    answer["undeclared"] = undeclared_numbers(answer)
    if answer["undeclared"]:
        answer["limitations"] = (
            (answer.get("limitations") or "")
            + " %d number(s) in the summary were not declared as claims and "
              "could not be checked." % len(answer["undeclared"])).strip()

    return answer


def check_totals(answer):
    """Shares should add to 100%. Parts should add to the stated total."""
    problems = []
    claims = answer.get("claims", [])

    shares = [c for c in claims if c.get("calc") == "share" and c.get("value") is not None]
    if len(shares) > 1:
        total = sum(float(c["value"]) for c in shares)
        if not about_equal(total, 100.0, 0.02):
            problems.append("The shares add up to %.1f%%, not 100%%." % total)

    totals = [c for c in claims if c.get("calc") == "sum" and c.get("value") is not None]
    parts = [c for c in claims
             if c.get("calc") in (None, "none") and c.get("value") is not None
             and c.get("unit") not in (None, "%")]
    for total_claim in totals:
        if not parts:
            continue
        added = sum(float(c["value"]) for c in parts)
        if not about_equal(added, float(total_claim["value"]), 0.02):
            problems.append("The parts add up to %.2f but the total says %.2f."
                            % (added, float(total_claim["value"])))
    return problems


# Numbers we ignore: years, and small whole numbers like "five products"
# or "26 trading days". Tune this if it turns out to be too noisy.
IGNORE_INTEGERS_BELOW = 100


def find_numbers(text):
    """Pull every number out of a sentence."""
    out = []
    for token in re.findall(r"-?\d[\d,]*\.?\d*", text or ""):
        try:
            out.append(float(token.replace(",", "")))
        except ValueError:
            pass
    return out


def undeclared_numbers(answer):
    """Numbers in the findings that the model never declared as a claim.

    This is the hole checks 1-3 cannot see. If the model writes a total in the
    paragraph but leaves it out of the claims list, nothing checks it - so we
    look for numbers in the text that are neither a claim nor in the log.
    """
    log_numbers = [v for _, v in numbers_in_log(answer.get("log", []))]
    declared = [c.get("value") for c in answer.get("claims", [])
                if c.get("value") is not None]

    out = []
    for n in find_numbers(answer.get("findings", "")):
        if float(n).is_integer() and 1900 <= n <= 2100:
            continue                                     # a year
        if float(n).is_integer() and abs(n) < IGNORE_INTEGERS_BELOW:
            continue                                     # a small count
        if any(about_equal(n, d) for d in declared):
            continue                                     # declared as a claim
        if any(about_equal(n, v) for v in log_numbers):
            continue                                     # came from a query
        out.append(n)
    return out


def write_feedback(answer):
    """What we send back to the model when something failed. Empty = all good."""
    bad = [c for c in answer.get("claims", [])
           if c.get("status") in ("unsupported", "flagged")]
    if not bad:
        return ""

    lines = ["Some numbers did not pass checking. Fix them and answer again.", ""]
    for c in bad:
        lines.append('- "%s" -> %s: %s' % (c.get("text"), c.get("status"), c.get("problem")))
    lines += ["",
              "Only use numbers that a query actually returned.",
              "For calculated numbers, fill in calc and inputs so we can check the maths."]
    return "\n".join(lines)


def tier3(question, max_retries=2):
    """Tier 2, but every number gets checked before the user sees it."""
    log = []
    start = time.time()
    feedback = ""
    answer = None

    for attempt in range(max_retries + 1):
        answer = tier2(question, log, feedback)
        answer = verify(answer)
        answer["retries"] = attempt
        feedback = write_feedback(answer)
        if feedback == "":
            break

    answer["tier"] = 3
    answer["seconds"] = round(time.time() - start, 1)

    hidden = len([c for c in answer["claims"] if c.get("status") == "unsupported"])
    if hidden:
        note = "%d claim(s) could not be verified and were hidden." % hidden
        answer["limitations"] = ((answer.get("limitations") or "") + " " + note).strip()
    return answer


# =====================================================================
# 5. THE VERIFIER  (this is our contribution - no AI in here)
# =====================================================================


def claims_to_show(answer):
    """Unsupported claims are hidden from the user."""
    return [c for c in answer.get("claims", []) if c.get("status") != "unsupported"]


def clean_findings(answer):
    """Take unverified numbers OUT OF THE PARAGRAPH, not just the claim list.

    Two kinds get replaced: claims that failed checking, and numbers that were
    never declared as claims at all (check 4). Hiding a claim while leaving the
    made-up number in the text is pointless - the paragraph is what people read.
    """
    text = answer.get("findings", "") or ""

    values = [c["value"] for c in answer.get("claims", [])
              if c.get("status") != "supported" and c.get("value") is not None]
    values += answer.get("undeclared", [])

    patterns = []
    for v in values:
        v = float(v)
        patterns += ["{:,.2f}".format(v), "{:,.1f}".format(v), "{:,.0f}".format(v),
                     "{:.2f}".format(v), "{:.1f}".format(v), "{:g}".format(v)]

    for p in sorted(set(patterns), key=len, reverse=True):   # longest first
        text = text.replace(p, "[unverified]")
    return text


def score(answer):
    """The numbers the evaluation notebook needs."""
    claims = answer.get("claims", [])
    n = len(claims)
    supported = len([c for c in claims if c.get("status") == "supported"])
    flagged = len([c for c in claims if c.get("status") == "flagged"])
    hidden = len([c for c in claims if c.get("status") == "unsupported"])
    calls = answer.get("log", [])
    return {"claims": n,
            "supported": supported,
            "flagged": flagged,
            "unsupported": hidden,
            "undeclared": len(answer.get("undeclared", [])),
            "evidence_coverage": supported / n if n else 0,
            "unsupported_rate": hidden / n if n else 0,
            "queries": len(calls),
            "queries_ok": len([c for c in calls if c["ok"]]) / len(calls) if calls else None}


def show(answer):
    """Print an answer the way the user would see it."""
    icons = {"supported": "[OK]  ", "flagged": "[WARN]", "unsupported": "[HIDE]"}
    print(clean_findings(answer))
    print()
    for c in claims_to_show(answer):
        print(icons.get(c.get("status"), "      "), c.get("text"))
        if c.get("problem"):
            print("         !", c["problem"])
        elif c.get("evidence"):
            print("         evidence:", c["evidence"].strip().replace("\n", " ")[:70])

    hidden = len(answer.get("claims", [])) - len(claims_to_show(answer))
    if hidden:
        print("\n%d claim(s) hidden - could not be verified." % hidden)
    if answer.get("undeclared"):
        print("%d number(s) in the summary were never declared as claims."
              % len(answer["undeclared"]))
    if answer.get("limitations"):
        print("\nLimitations:", answer["limitations"])
    if answer.get("insufficient_data"):
        print("\nThe data cannot answer this question.")
    print("\ntier %s | %s retries | %s seconds"
          % (answer.get("tier"), answer.get("retries"), answer.get("seconds")))


In [ ]:
import yaml
import pandas as pd
import matplotlib.pyplot as plt

questions = yaml.safe_load(open('../eval/benchmark.yaml'))['questions']
print(len(questions), 'questions')


## 1. Run everything


In [ ]:
rows = []

for i, q in enumerate(questions, 1):
    for tier, fn in [(1, tier1), (2, tier2), (3, tier3)]:
        print('[%2d/15] %s tier %d' % (i, q['id'], tier), end=' ')
        try:
            a = fn(q['question'])
            s = score(a)
            rows.append({
                'id': q['id'],
                'tier': tier,
                'difficulty': q['difficulty'],
                'answerable': q['answerable'],
                'gt_value': q.get('gt_value'),
                'also_accept': q.get('also_accept'),
                'values': [c.get('value') for c in a['claims']
                           if c.get('value') is not None],
                'refused': a['insufficient_data'],
                'claims': s['claims'],
                'supported': s['supported'],
                'unsupported_rate': s['unsupported_rate'],
                'evidence_coverage': s['evidence_coverage'],
                'queries_ok': s['queries_ok'],
                'retries': a['retries'],
                'seconds': a['seconds'],
            })
            print('ok')
        except Exception as e:
            print('ERROR', e)

df = pd.DataFrame(rows)
df.to_csv('../eval/results/all_runs.csv', index=False)
print('saved', len(df), 'rows')


## 2. Was the answer right?

Correct means any number the tier gave is within 1% of the answer we worked
out ourselves in SQL.


In [ ]:
def is_correct(row):
    """Correct if any number the tier gave matches the answer we worked out.

    Some questions have two right answers - "by what percentage" can also be
    answered in pounds - so also_accept holds the alternatives.
    """
    if row['gt_value'] is None or pd.isna(row['gt_value']):
        return None                     # the trick questions have no number

    targets = [float(row['gt_value'])]
    targets += [float(x) for x in (row.get('also_accept') or [])]

    for v in row['values']:
        for t in targets:
            if about_equal(float(v), t):
                return 1.0
    return 0.0


df['correct'] = df.apply(is_correct, axis=1)

answerable = df[df.answerable].copy()
tricks = df[~df.answerable].copy()

df[['id', 'tier', 'gt_value', 'values', 'correct']].head(12)


## 3. The scoreboard


In [ ]:
summary = pd.DataFrame({
    'accuracy':          answerable.groupby('tier').correct.mean(),
    'queries_ok':        df.groupby('tier').queries_ok.mean(),
    'evidence_coverage': df.groupby('tier').evidence_coverage.mean(),
    'unsupported_rate':  df.groupby('tier').unsupported_rate.mean(),
    'refused_tricks':    tricks.groupby('tier').refused.sum(),
    'avg_retries':       df.groupby('tier').retries.mean(),
    'avg_seconds':       df.groupby('tier').seconds.mean(),
}).round(3)

summary.to_csv('../eval/results/summary.csv')
summary


## 4. Accuracy by difficulty


In [ ]:
(answerable.pivot_table(index='difficulty', columns='tier',
                       values='correct', aggfunc='mean')
           .reindex(['easy', 'medium', 'hard'])
           .round(2))


## 5. The three impossible questions

The clearest table in the project. Anything that is not a refusal is a
made-up number a manager would have acted on.


In [ ]:
tricks.pivot_table(index='id', columns='tier', values='refused')


## 6. Charts for the report


In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 3.5))

answerable.groupby('tier').correct.mean().plot(
    kind='bar', ax=ax[0], color='steelblue')
ax[0].set_title('Accuracy')
ax[0].set_ylim(0, 1)

df.groupby('tier').unsupported_rate.mean().plot(
    kind='bar', ax=ax[1], color='indianred')
ax[1].set_title('Made-up claims (lower is better)')

df.groupby('tier').seconds.mean().plot(
    kind='bar', ax=ax[2], color='grey')
ax[2].set_title('Seconds per answer')

for a in ax:
    a.set_xlabel('tier')

plt.tight_layout()
plt.savefig('../eval/results/comparison.png', dpi=150)
plt.show()


## 7. What this shows

Fill in once you have the real numbers:

- Tier 1 to Tier 2: accuracy went from ___ to ___ (what using the real data
  is worth)
- Tier 2 to Tier 3: made-up claims went from ___ to ___ (what the checking
  is worth)
- Tier 3 took ___ seconds against Tier 2's ___

**Say the cost out loud.** "Slower but much more accurate" is a believable
finding. Pretending there is no trade-off is not.
